# Notebook 06 — Price Prediction Model

**Member 3 task B** — Train a listing-level price model on Barcelona data (London listings not in repo due to size). Report RMSE, MAE, R² and feature importances.

This is the KPMG slide requirement ('Price Prediction Modeling + Evaluation and Metrics').

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from src.data_io import PROCESSED_DIR

RANDOM_STATE = 42
MODELS_DIR = Path('..') / 'models'
FIGURES_DIR = Path('..') / 'reports' / 'figures' / 'clustering'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK")

## 1. Load listing features

In [ ]:
bcn = pd.read_csv(PROCESSED_DIR / 'barcelona' / 'barcelona_listings_features.csv')
print(f"Barcelona features: {bcn.shape}")

# Note: London listings not in repo (size limit). Price model uses Barcelona only.
# The chatbot's pricing queries are served from neighbourhood_kpis median_nightly_price,
# so the listing-level model is the KPMG slide evaluation artefact.
features_all = bcn.copy()
print(f"Combined dataset: {features_all.shape}")

## 2. Target & feature selection

In [ ]:
TARGET = 'ttm_avg_rate'

# Keep rows with a valid (non-zero) target
df = features_all[features_all[TARGET] > 0].copy()
print(f"Rows with valid price: {len(df)} / {len(features_all)}")

NUMERIC_FEATURES = ['guests', 'bedrooms', 'beds', 'baths',
                    'host_listing_count', 'num_reviews']
CATEGORICAL_FEATURES = ['room_type', 'host_commercial_tier', 'occupancy_band', 'price_band']
BOOL_FEATURES = ['entire_home_flag', 'multi_listing_host', 'host_5_plus_listings',
                 'has_registration', 'superhost']
GEO_FEATURE = ['geo_key']  # one-hot encoded subdivision

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES + BOOL_FEATURES + GEO_FEATURE

# Fill missing
for col in NUMERIC_FEATURES:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(df[col].median())
for col in BOOL_FEATURES:
    df[col] = df[col].astype(bool).astype(int)
for col in CATEGORICAL_FEATURES + GEO_FEATURE:
    df[col] = df[col].fillna('unknown')

X = df[ALL_FEATURES]
y = df[TARGET]
print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"y stats: min={y.min():.1f}, median={y.median():.1f}, max={y.max():.1f}")

## 3. Train / test split (80/20, stratified by city)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train: {len(X_train)}  Test: {len(X_test)}")

## 4. Preprocessing pipeline

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), NUMERIC_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
     CATEGORICAL_FEATURES + GEO_FEATURE),
    ('bool', 'passthrough', BOOL_FEATURES),
])

# Fit on train only
preprocessor.fit(X_train)
X_train_t = preprocessor.transform(X_train)
X_test_t  = preprocessor.transform(X_test)
print(f"Transformed train shape: {X_train_t.shape}")

## 5. Train three models

In [ ]:
models = {
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest':    RandomForestRegressor(n_estimators=200, max_depth=12,
                                              random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                                    learning_rate=0.05, random_state=RANDOM_STATE),
}

results = {}
for name, mdl in models.items():
    mdl.fit(X_train_t, y_train)
    preds = mdl.predict(X_test_t)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae  = mean_absolute_error(y_test, preds)
    r2   = r2_score(y_test, preds)
    results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'model': mdl, 'preds': preds}
    print(f"{name:25s} | RMSE={rmse:6.1f} | MAE={mae:6.1f} | R²={r2:.3f}")

## 6. Select best model & save

In [ ]:
best_name = min(results, key=lambda n: results[n]['RMSE'])
best = results[best_name]
print(f"Best model: {best_name}")
print(f"  RMSE={best['RMSE']:.1f}  MAE={best['MAE']:.1f}  R²={best['R2']:.3f}")

# Save the full pipeline (preprocessor + model)
model_bundle = {'preprocessor': preprocessor, 'model': best['model'],
                'feature_names': ALL_FEATURES, 'model_name': best_name}
model_path = MODELS_DIR / 'price_model.joblib'
joblib.dump(model_bundle, model_path)
print(f"Saved model bundle → {model_path}")

## 7. Feature importance plot

In [ ]:
# Build feature names after one-hot
cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(
    CATEGORICAL_FEATURES + GEO_FEATURE
).tolist()
all_feature_names = NUMERIC_FEATURES + cat_feature_names + BOOL_FEATURES

mdl = best['model']
if hasattr(mdl, 'feature_importances_'):
    importances = mdl.feature_importances_
    fi = pd.Series(importances, index=all_feature_names).sort_values(ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(9, 6))
    fi.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_xlabel('Feature importance')
    ax.set_title(f'Top 20 feature importances — {best_name}')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'price_model_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved price_model_feature_importance.png")
else:
    print(f"Note: {best_name} does not expose feature_importances_; skipping plot.")

## 8. Predicted vs actual plot

In [ ]:
preds = best['preds']
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, preds, alpha=0.3, s=12, color='steelblue')
lim = max(y_test.max(), preds.max()) * 1.05
ax.plot([0, lim], [0, lim], 'r--', linewidth=1.5, label='Perfect fit')
ax.set_xlabel('Actual nightly price (€)')
ax.set_ylabel('Predicted nightly price (€)')
ax.set_title(f'Predicted vs Actual — {best_name}\nRMSE={best["RMSE"]:.1f}  MAE={best["MAE"]:.1f}  R²={best["R2"]:.3f}')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'price_model_pred_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved price_model_pred_vs_actual.png")

## 9. Append metrics to model_metrics.md

In [ ]:
metrics_path = Path('..') / 'reports' / 'model_metrics.md'

# Build results table
rows = []
for name, res in results.items():
    rows.append({'Model': name, 'RMSE': f"{res['RMSE']:.1f}",
                 'MAE': f"{res['MAE']:.1f}", 'R²': f"{res['R2']:.3f}"})
results_df = pd.DataFrame(rows)

price_block = f"""

## Price Prediction Model

| Model | RMSE | MAE | R² |
|---|---|---|---|
{chr(10).join('| ' + ' | '.join(str(v) for v in row.values()) + ' |' for _, row in results_df.iterrows())}

**Best model:** {best_name} (lowest RMSE on 20% held-out test set)

### Training details
- Target: `ttm_avg_rate` (trailing 12-month average nightly rate, €)
- Training set: {len(X_train)} listings | Test set: {len(X_test)} listings
- City: Barcelona (London listing-level data excluded from repo due to size)
- Numeric features: `{', '.join(NUMERIC_FEATURES)}`
- Categorical features (one-hot): `{', '.join(CATEGORICAL_FEATURES + GEO_FEATURE)}`
- Boolean flags: `{', '.join(BOOL_FEATURES)}`
- Saved model: `models/price_model.joblib`

"""

with open(metrics_path, 'a') as f:
    f.write(price_block)
print(f"Appended price model metrics to {metrics_path}")

In [ ]:
print("\n✅ Notebook 06 complete.")
print(f"   Best model: {best_name}")
print(f"   RMSE={best['RMSE']:.1f}  MAE={best['MAE']:.1f}  R²={best['R2']:.3f}")
print(f"   Saved: models/price_model.joblib")